# 07 Build Baseline Features

Build dense baseline features, Markov transition features, and SVD embeddings.

In [1]:
from pathlib import Path
import dataclasses
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
PROJECT_ROOT

PosixPath('/root/projects/bert4rec')

In [2]:
from src.baselines import build_baseline_features, load_baseline_config

base_config = load_baseline_config(PROJECT_ROOT / "configs" / "baselines.yaml")
base_config

BaselineConfig(input_prefixes={'train': 'data/processed/train_prefixes.parquet', 'valid': 'data/processed/valid_prefixes.parquet', 'test': 'data/processed/test_prefixes.parquet'}, outputs={'features': 'data/exports/baseline_features.parquet', 'embeddings': 'data/exports/baseline_embeddings_128d.parquet', 'markov_stats': 'artifacts/baselines/markov_transition_stats.parquet', 'manifest': 'artifacts/manifests/baseline_manifest.json'}, svd_dim=128, hash_features=262144, top_markov_k=10, last_k=5, random_state=42, dry_run_rows=None)

In [3]:
DRY_RUN = False
config = base_config
if DRY_RUN:
    config = dataclasses.replace(config, dry_run_rows=512, svd_dim=32, hash_features=32768)
print(config)

BaselineConfig(input_prefixes={'train': 'data/processed/train_prefixes.parquet', 'valid': 'data/processed/valid_prefixes.parquet', 'test': 'data/processed/test_prefixes.parquet'}, outputs={'features': 'data/exports/baseline_features.parquet', 'embeddings': 'data/exports/baseline_embeddings_128d.parquet', 'markov_stats': 'artifacts/baselines/markov_transition_stats.parquet', 'manifest': 'artifacts/manifests/baseline_manifest.json'}, svd_dim=128, hash_features=262144, top_markov_k=10, last_k=5, random_state=42, dry_run_rows=None)


In [4]:
manifest = build_baseline_features(config, PROJECT_ROOT)
manifest

Dense baseline features:   0%|          | 0/139123 [00:00<?, ?it/s]

{'created_at': '2026-05-10T13:52:35.100078+00:00',
 'rows': 139123,
 'dry_run_rows': None,
 'svd_dim': 128,
 'hash_features': 262144,
 'svd_explained_variance_ratio_sum': 0.9999996624681792,
 'outputs': {'features': 'data/exports/baseline_features.parquet',
  'embeddings': 'data/exports/baseline_embeddings_128d.parquet',
  'markov_stats': 'artifacts/baselines/markov_transition_stats.parquet',
  'manifest': 'artifacts/manifests/baseline_manifest.json'},
 'numeric_feature_columns': ['prefix_len',
  'num_events_total',
  'unique_event_count',
  'event_entropy',
  'repeat_count',
  'repeat_rate',
  'session_boundary_count',
  'gap_mean',
  'gap_std',
  'gap_max',
  'markov_last_total_count',
  'markov_top1_prob',
  'markov_top10_entropy',
  'last_event_1',
  'last_event_2',
  'last_event_3',
  'last_event_4',
  'last_event_5'],
 'markov_rows': 4962}

In [5]:
import pandas as pd
features = pd.read_parquet(PROJECT_ROOT / config.outputs["features"])
emb = pd.read_parquet(PROJECT_ROOT / config.outputs["embeddings"])
print(features.shape, emb.shape)
features.head()

(139123, 29) (139123, 9)


,user_id,split,prefix_len,num_events_total,prefix_start_ts,prefix_end_ts,next_event_token_id,label_available_retention_7d,label_retention_7d,label_available_retention_14d,...,last_event_1,last_event_2,last_event_3,last_event_4,last_event_5,markov_last_total_count,markov_top1_prob,markov_top10_entropy,markov_actual_prob,markov_actual_rank
0,10000235382394935150,train,50,75,1773673777,1773673783,62.0,True,0.0,False,...,61,60,59,189,178,3401.0,0.987651,0.082264,0.987651,1.0
1,10000279983408954705,train,50,406,1774705665,1774705676,407.0,False,NaN,False,...,389,375,360,336,270,676.0,0.677515,0.960153,0.677515,1.0
2,10000279983408954705,train,100,406,1774705665,1774705707,77.0,False,NaN,False,...,76,71,70,69,75,488.0,0.965164,0.174158,0.965164,1.0
3,10000279983408954705,train,150,406,1774705665,1774714019,74.0,False,NaN,False,...,73,72,67,66,65,493.0,0.967546,0.155401,0.967546,1.0
4,10000413951987079100,train,50,707,1772150043,1772150050,60.0,True,0.0,True,...,59,212,58,57,189,1737.0,0.984456,0.097999,0.984456,1.0
